<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Breakout/Resistance_Breakout_Daily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Resistance Breakout Strategy & Performance Scanner

This notebook implements a technical analysis scanner designed to identify **Resistance Breakouts**. The strategy focuses on identifying stocks that establish a clear price ceiling and subsequently break through it with high momentum.

**Key Features:**
* **Dynamic Resistance Detection:** Identifies price levels where a ticker has 'touched' or peaked multiple times (configurable via `min_touches`) within a lookback window.
* **Breakout Validation:** Triggers a signal only when the price closes above the resistance level by a specified percentage buffer and sets a new local high.
* **Ticker Locking:** Prevents redundant signals by enforcing a 'cool-off' period (e.g., 10 days) after a breakout is detected for a specific ticker.
* **Forward Performance Tracking:** Automatically calculates the 1-day through 5-day returns following each breakout and tracks the 'Success Rate' (percentage of time the price remains above the breakout level).
* **Buffered Data Ingestion:** Automatically downloads an extra 10 days of historical data beyond the scan window to ensure end-of-year signals have complete performance data.

**Current Configuration:** Scanning high-volume tickers for breakouts between January 2025 and December 2025.

In [9]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

In [10]:
# Clear all DataFrames from memory
import gc

# Get a list of all variables in the global namespace
all_vars = list(globals().keys())

# Identify and delete pandas DataFrames
for var_name in all_vars:
    if isinstance(globals()[var_name], pd.DataFrame):
        del globals()[var_name]
        print(f"Deleted DataFrame: {var_name}")

# Run garbage collector to free up memory
gc.collect()

print("All DataFrames cleared from memory.")

Deleted DataFrame: csv_input
Deleted DataFrame: full_historical_data
Deleted DataFrame: intraday_df
Deleted DataFrame: watchlist_df
Deleted DataFrame: performance_df
Deleted DataFrame: valid_subset
All DataFrames cleared from memory.


In [11]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

### 1. Configuration and Data Ingestion
This cell defines the date range for the scan and calculates a 10-day 'buffer' for `DATA_END_DATE` to ensure we can track performance for year-end breakouts. It then loads the ticker list from Google Drive and downloads the necessary historical data via `yfinance`.

In [12]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

# 1. Parameterize scan range
# Set this to a specific date (e.g., '2026-06-26') for reproducible results, or leave as None for today.
SCAN_DATE_STR = None # Set to 'YYYY-MM-DD' for a specific historical scan date, or None for today.

# Data download buffer for forward performance tracking (as per notebook description)
DATA_DOWNLOAD_BUFFER_DAYS = 10

if SCAN_DATE_STR:
    # If SCAN_DATE_STR is provided, use it as the fixed scan date
    SCAN_DATE_DT = pd.to_datetime(SCAN_DATE_STR)
else:
    # Otherwise, use the current date as the scan date
    SCAN_DATE_DT = datetime.now() # Using datetime.now() directly for accuracy

# The end date for yfinance download must be one day *after* the last desired data point.
# This ensures we get data up to and including SCAN_DATE_DT + DATA_DOWNLOAD_BUFFER_DAYS.
DATA_DOWNLOAD_END_DT = SCAN_DATE_DT + timedelta(days=DATA_DOWNLOAD_BUFFER_DAYS + 1) # +1 for yfinance exclusive end

# Format dates for use in print statements and yfinance calls
SCAN_DATE = SCAN_DATE_DT.strftime('%Y-%m-%d')
DATA_DOWNLOAD_END = DATA_DOWNLOAD_END_DT.strftime('%Y-%m-%d')

# Data download should cover at least the lookback window (60 days) + scan window (30 days).
# Using 180 calendar days to be safe for a 60-day lookback window.
START_DATA_DOWNLOAD_DT = SCAN_DATE_DT - timedelta(days=180)
START_DATA_DOWNLOAD = START_DATA_DOWNLOAD_DT.strftime('%Y-%m-%d')

# Identify breakouts in the last 30 trading days (including SCAN_DATE)
START_BREAKOUT_IDENTIFICATION_DT = SCAN_DATE_DT - timedelta(days=30)
START_BREAKOUT_IDENTIFICATION = START_BREAKOUT_IDENTIFICATION_DT.strftime('%Y-%m-%d')

# Print statements reflecting the ranges
print(f"Scan will conclude on: {SCAN_DATE}")
print(f"Historical data download range: {START_DATA_DOWNLOAD} to {DATA_DOWNLOAD_END} (includes {DATA_DOWNLOAD_BUFFER_DAYS} day buffer for performance tracking)")
print(f"Breakout identification window: {START_BREAKOUT_IDENTIFICATION} to {SCAN_DATE}")

# 2. Ingest list of tickers from your CSV file
tickers = []
try:
    csv_input = pd.read_csv(OptionVolume200)
    tickers = csv_input['Symbol'].dropna().unique().tolist()
    tickers = [str(t).strip().upper() for t in tickers if str(t).isalpha()]
except Exception as e:
    print(f"Error reading CSV: {e}. Falling back to test list.")
    tickers = ["AAPL", "MSFT", "AMD", "NVDA", "INTC", "TSLA"]

# 3. Batch download historical data once
full_historical_data = pd.DataFrame()
if tickers:
    print(f"Downloading historical daily data for {len(tickers)} tickers from {START_DATA_DOWNLOAD} to {DATA_DOWNLOAD_END}...")
    try:
        full_historical_data = yf.download(tickers, start=START_DATA_DOWNLOAD, end=DATA_DOWNLOAD_END, group_by='ticker', auto_adjust=True)
        print("Historical daily data download complete.")

        # Only attempt intraday data augmentation if SCAN_DATE is today
        if SCAN_DATE_DT.date() == datetime.now().date():
            print(f"Augmenting with current day's intraday data ({datetime.now().strftime('%H:%M')} - 5-minute interval) for {SCAN_DATE}...")
            current_date_dt = pd.to_datetime(SCAN_DATE)

            # Iterate through tickers to fetch intraday data individually
            for ticker in tickers:
                if (ticker, 'Close') not in full_historical_data.columns:
                    continue

                try:
                    intraday_df = yf.download(ticker, period='1d', interval='5m', auto_adjust=True, progress=False)
                    if not intraday_df.empty:
                        latest_intraday_close = intraday_df['Close'].iloc[-1]
                        latest_intraday_high = intraday_df['High'].iloc[-1]
                        latest_intraday_low = intraday_df['Low'].iloc[-1]

                        if current_date_dt in full_historical_data.index:
                            full_historical_data.loc[current_date_dt, (ticker, 'Close')] = latest_intraday_close
                            if latest_intraday_high > full_historical_data.loc[current_date_dt, (ticker, 'High')]:
                                full_historical_data.loc[current_date_dt, (ticker, 'High')] = latest_intraday_high
                            if latest_intraday_low < full_historical_data.loc[current_date_dt, (ticker, 'Low')]:
                                full_historical_data.loc[current_date_dt, (ticker, 'Low')] = latest_intraday_low
                except Exception as e:
                    pass
            print("Intraday data augmentation complete.")
        else:
            print(f"Skipping intraday data augmentation as SCAN_DATE ({SCAN_DATE}) is not today.")

    except Exception as e:
        print(f"Error fetching daily data: {e}")

if full_historical_data.empty:
    print("No historical data available for analysis.")

Scan will conclude on: 2026-06-27
Historical data download range: 2025-12-29 to 2026-07-08 (includes 10 day buffer for performance tracking)
Breakout identification window: 2026-05-28 to 2026-06-27


[                       0%                       ]

[*********************100%***********************]  199 of 199 completed


Historical daily data download complete.
Augmenting with current day's intraday data (16:34 - 5-minute interval) for 2026-06-27...


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SATS']: YFPricesMissingError('possibly delisted; no price data found  (period=1d)')


Intraday data augmentation complete.


### 2. Resistance Breakout Scanner
This function iterates through the historical data to find price levels where a stock has peaked multiple times. It triggers a signal when the price decisively closes above that resistance level. It also includes a 'cool-off' mechanism to prevent redundant signals for the same stock.

In [13]:
def scan_resistance_breakout(ticker_list, full_data,
                             start_date_str, end_date_str,
                             resistance_buffer_pct=0.005, min_touches=3,
                             lookback_window=60, breakout_buffer_pct=0.0025,
                             cool_off_days=30):
    """
    Scans for unique resistance breakouts between start_date_str and end_date_str.
    """
    candidates = []
    last_breakout_date = {}

    # Convert strings to datetime objects for comparison
    start_dt = pd.to_datetime(start_date_str)
    end_dt = pd.to_datetime(end_date_str)

    print(f"Scanning {len(ticker_list)} tickers for triggers between {start_date_str} and {end_dt.strftime('%Y-%m-%d')}...")

    for ticker in ticker_list:
        try:
            if ticker not in full_data.columns.levels[0]:
                continue
            df = full_data[ticker].dropna().copy()
            if df.empty or len(df) < lookback_window + 2:
                continue

            last_breakout_date[ticker] = None

            for i in range(lookback_window, len(df)):
                current_date = df.index[i]

                # Filter: Only identify triggers within the defined scan window
                if current_date < start_dt or current_date > end_dt:
                    continue

                # Check if we are within the cool-off period
                if last_breakout_date[ticker] is not None:
                    days_since = (current_date - last_breakout_date[ticker]).days
                    if days_since < cool_off_days:
                        continue

                recent_df = df.iloc[i-lookback_window : i]
                current_close = df['Close'].iloc[i]
                previous_close = df['Close'].iloc[i-1]
                highest_high_in_window = recent_df['High'].max()

                # 1. Identify Resistance Levels
                highs = recent_df['High'].values
                sorted_highs = np.sort(highs)
                identified_levels = []

                for val in sorted_highs:
                    cluster = [h for h in highs if abs(h - val) / val <= resistance_buffer_pct]
                    if len(cluster) >= min_touches:
                        res_level = np.mean(cluster)
                        if not any(abs(res_level - lvl) / lvl < 0.02 for lvl, _ in identified_levels):
                            identified_levels.append((res_level, len(cluster)))

                # 2. Check Breakout Conditions
                for res_price, touches in identified_levels:
                    breakout_threshold = res_price * (1 + breakout_buffer_pct)

                    # Combined condition: Close above resistance AND new high for the lookback window
                    if (current_close > breakout_threshold) and \
                       (previous_close <= breakout_threshold) and \
                       (current_close > highest_high_in_window):
                            candidates.append({
                                "Ticker": ticker,
                                "Date": current_date.strftime('%Y-%m-%d'),
                                "Res_Level": round(res_price, 2),
                                "Breakout_Threshold": round(breakout_threshold, 2),
                                "Touches": touches,
                                "Close": round(current_close, 2),
                                "Prev_High": round(highest_high_in_window, 2)
                            })
                            last_breakout_date[ticker] = current_date
                            break

        except Exception:
            continue

    return pd.DataFrame(candidates)

# Re-run the scan using the parameter window
if 'full_historical_data' in globals() and not full_historical_data.empty:
    watchlist_df = scan_resistance_breakout(
        tickers,
        full_historical_data,
        start_date_str=START_BREAKOUT_IDENTIFICATION,
        end_date_str=SCAN_DATE, # Use SCAN_DATE here
        min_touches=5,
        cool_off_days=5
    )
    print(f"\nTotal triggers identified: {len(watchlist_df)}")
    display(watchlist_df.head(20))

Scanning 199 tickers for triggers between 2026-05-28 and 2026-06-27...

Total triggers identified: 19


,Ticker,Date,Res_Level,Breakout_Threshold,Touches,Close,Prev_High
0,MSFT,2026-05-29,428.69,429.76,5,450.24,432.76
1,AAPL,2026-06-02,312.04,312.82,5,315.20,315.00
2,IWM,2026-06-12,289.23,289.95,5,292.26,292.19
3,AVGO,2026-05-29,433.14,434.22,6,446.06,441.66
4,UNH,2026-06-25,407.45,408.47,5,415.53,414.16
5,LLY,2026-06-26,1169.48,1172.41,5,1208.12,1182.73
6,BE,2026-06-18,296.46,297.20,5,328.91,322.83
7,JPM,2026-06-12,315.65,316.44,13,320.72,320.24
8,SNOW,2026-05-28,177.77,178.22,5,239.20,184.74
9,C,2026-06-04,131.44,131.77,5,135.15,134.65


### 3. Performance Analysis and Validation
This final section calculates the forward returns for 1 to 5 days following every identified breakout. It generates a summary table showing the average returns and a 'Success Rate,' verifying how often the price stayed above the breakout level.

In [14]:
def calculate_post_breakout_performance(watchlist, full_data):
    results = []

    for _, row in watchlist.iterrows():
        ticker = row['Ticker']
        breakout_date = row['Date']
        res_level = row['Res_Level']
        breakout_close = row['Close']

        if ticker not in full_data.columns.levels[0]:
            continue

        df = full_data[ticker].dropna()
        try:
            idx = df.index.get_loc(breakout_date)
        except KeyError:
            continue

        performance = row.to_dict()

        # Loop specifically for Days 1 through 5
        for d in [1, 2, 3, 4, 5]:
            future_idx = idx + d
            if future_idx < len(df):
                future_close = df['Close'].iloc[future_idx]
                ret = ((future_close - breakout_close) / breakout_close) * 100
                performance[f'Day_{d}_Return_%'] = round(ret, 2)
                performance[f'Day_{d}_Above_Res'] = bool(future_close >= res_level)
            else:
                performance[f'Day_{d}_Return_%'] = np.nan
                performance[f'Day_{d}_Above_Res'] = None

        results.append(performance)

    return pd.DataFrame(results)

if 'watchlist_df' in globals() and not watchlist_df.empty:
    performance_df = calculate_post_breakout_performance(watchlist_df, full_historical_data)

    # Calculate Summary for Days 1-5
    summary_data = {}
    validation_log = []

    for d in [1, 2, 3, 4, 5]:
        col_name = f'Day_{d}_Above_Res'
        valid_subset = performance_df[performance_df[col_name].notnull()]

        total_valid = len(valid_subset)
        success_count = valid_subset[col_name].sum() if total_valid > 0 else 0
        fail_count = total_valid - success_count

        avg_ret = valid_subset[f'Day_{d}_Return_%'].mean() if total_valid > 0 else np.nan
        pct_above = (success_count / total_valid * 100) if total_valid > 0 else 0

        summary_data[f'Day_{d} Avg Return %'] = avg_ret
        summary_data[f'Day_{d} % Above Res'] = pct_above

        validation_log.append({"Day": d, "Total": total_valid, "Above": success_count, "Below": fail_count})

    print("--- Updated Performance Summary (with 10-day buffer) ---")
    display(pd.Series(summary_data).to_frame(name='Value'))

    print("\n--- Raw Verification Counts ---")
    display(pd.DataFrame(validation_log))

    print("\n--- Detailed Performance Table (Full) ---")
    # Sort by Date in descending order before displaying
    performance_df['Date'] = pd.to_datetime(performance_df['Date'])
    display(performance_df.sort_values(by='Date', ascending=False))

--- Updated Performance Summary (with 10-day buffer) ---


,Value
Day_1 Avg Return %,1.280000
Day_1 % Above Res,94.444444
Day_2 Avg Return %,0.820000
Day_2 % Above Res,87.500000
Day_3 Avg Return %,0.373750
Day_3 % Above Res,75.000000
Day_4 Avg Return %,-1.498125
Day_4 % Above Res,68.750000
Day_5 Avg Return %,-2.870625
Day_5 % Above Res,68.750000



--- Raw Verification Counts ---


,Day,Total,Above,Below
0,1,18,17,1
1,2,16,14,2
2,3,16,12,4
3,4,16,11,5
4,5,16,11,5



--- Detailed Performance Table (Full) ---


,Ticker,Date,Res_Level,Breakout_Threshold,Touches,Close,Prev_High,Day_1_Return_%,Day_1_Above_Res,Day_2_Return_%,Day_2_Above_Res,Day_3_Return_%,Day_3_Above_Res,Day_4_Return_%,Day_4_Above_Res,Day_5_Return_%,Day_5_Above_Res
5,LLY,2026-06-26,1169.48,1172.41,5,1208.12,1182.73,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
4,UNH,2026-06-25,407.45,408.47,5,415.53,414.16,2.97,True,NaN,None,NaN,None,NaN,None,NaN,None
12,MRK,2026-06-25,121.40,121.70,9,125.45,123.11,2.56,True,NaN,None,NaN,None,NaN,None,NaN,None
6,BE,2026-06-18,296.46,297.20,5,328.91,322.83,5.15,True,-2.11,True,-0.83,True,-6.00,True,-23.38,False
16,EEM,2026-06-18,69.99,70.17,5,70.79,70.49,0.59,True,-5.11,False,-5.00,False,-4.00,False,-5.09,False
13,MRNA,2026-06-17,56.00,56.14,5,61.80,59.48,3.50,True,-3.97,True,-1.29,True,-2.23,True,-3.32,True
2,IWM,2026-06-12,289.23,289.95,5,292.26,292.19,0.81,True,-0.06,True,-0.81,True,1.14,True,2.03,True
17,ROKU,2026-06-12,127.24,127.56,6,143.66,133.46,-1.92,True,-3.97,True,-4.43,True,-3.89,True,-5.89,True
7,JPM,2026-06-12,315.65,316.44,13,320.72,320.24,-0.41,True,3.25,True,3.97,True,1.40,True,3.35,True
14,XLF,2026-06-12,52.42,52.55,10,53.15,53.00,0.42,True,1.90,True,1.34,True,0.44,True,1.03,True


In [15]:
print("\n--- Below Resistance Analysis ---")
below_res_analysis = []

for d in [1, 2, 3, 4, 5]:
    col_name = f'Day_{d}_Above_Res'

    # Filter out None values to get only True/False for the calculation
    valid_observations = performance_df[performance_df[col_name].notnull()]

    # Count 'False' values
    false_count = valid_observations[col_name].value_counts().get(False, 0)

    # Total for percentage calculation (only non-None values)
    total_for_percentage = len(valid_observations)

    # Calculate percentage
    percent_false = (false_count / total_for_percentage * 100) if total_for_percentage > 0 else 0

    below_res_analysis.append({
        "Day": d,
        "Count_Below_Resistance": false_count,
        "Percent_Below_Resistance": round(percent_false, 2)
    })

display(pd.DataFrame(below_res_analysis))


--- Below Resistance Analysis ---


,Day,Count_Below_Resistance,Percent_Below_Resistance
0,1,1,5.56
1,2,2,12.50
2,3,4,25.00
3,4,5,31.25
4,5,5,31.25
